In [ ]:
# CREATE NEO4J DOCKER CONTAINER
# docker run \
#   --name neo4j-db-local \
#   -p 7474:7474 \
#   -p 7687:7687 \
#   -e NEO4J_AUTH=neo4j/test12345 \
#   -v $HOME/neo4j-db-local/data:/data \
#   -v $HOME/neo4j-db-local/logs:/logs \
#   -d neo4j:latest

In [ ]:
# Neo4j adminsitration panel: http://localhost:7474

In [ ]:
# Cypher - populate database
# CREATE
#   (europe:Continent {name:'Europe', code:'EU', areaKm2:10180000, isGoodForLiving:True}),
#   (asia:Continent {name:'Asia', code:'AS', areaKm2:44580000}),
#   (northAmerica:Continent {name:'North America', code:'NA', areaKm2:24710000}),

#   (poland:Country {name:'Poland', code:'PL', population:37900000}),
#   (germany:Country {name:'Germany', code:'DE', population:83200000}),
#   (japan:Country {name:'Japan', code:'JP', population:123900000}),
#   (india:Country {name:'India', code:'IN', population:1428000000}),
#   (usa:Country {name:'United States', code:'US', population:339000000}),
#   (canada:Country {name:'Canada', code:'CA', population:40100000}),

#   (warsaw:City {name:'Warsaw', population:1860000, areaKm2:517}),
#   (krakow:City {name:'Krakow', population:804000, areaKm2:327}),
#   (berlin:City {name:'Berlin', population:3645000, areaKm2:891}),
#   (munich:City {name:'Munich', population:1490000, areaKm2:310}),
#   (tokyo:City {name:'Tokyo', population:13960000, areaKm2:2194}),
#   (osaka:City {name:'Osaka', population:2750000, areaKm2:225}),
#   (delhi:City {name:'Delhi', population:32900000, areaKm2:1484}),
#   (mumbai:City {name:'Mumbai', population:12400000, areaKm2:603}),
#   (newYork:City {name:'New York City', population:8468000, areaKm2:783}),
#   (chicago:City {name:'Chicago', population:2696000, areaKm2:589}),
#   (toronto:City {name:'Toronto', population:2930000, areaKm2:630}),
#   (vancouver:City {name:'Vancouver', population:706000, areaKm2:115}),
#   (losangeles:City {name:'Los Angeles', population:3970000, areaKm2:1304}),
    
#   (europe)-[:HAS_COUNTRY]->(poland),
#   (europe)-[:HAS_COUNTRY]->(germany),
#   (asia)-[:HAS_COUNTRY]->(japan),
#   (asia)-[:HAS_COUNTRY]->(india),
#   (northAmerica)-[:HAS_COUNTRY]->(usa),
#   (northAmerica)-[:HAS_COUNTRY]->(canada),

#   (poland)-[:HAS_CITY]->(warsaw),
#   (poland)-[:HAS_CITY]->(krakow),
#   (germany)-[:HAS_CITY]->(berlin),
#   (germany)-[:HAS_CITY]->(munich),
#   (japan)-[:HAS_CITY]->(tokyo),
#   (japan)-[:HAS_CITY]->(osaka),
#   (india)-[:HAS_CITY]->(delhi),
#   (india)-[:HAS_CITY]->(mumbai),
#   (usa)-[:HAS_CITY]->(newYork),
#   (usa)-[:HAS_CITY]->(chicago),
#   (usa)-[:HAS_CITY]->(losangeles),
#   (canada)-[:HAS_CITY]->(toronto),
#   (canada)-[:HAS_CITY]->(vancouver);

In [ ]:
# Cypher - create indexes

# CREATE CONSTRAINT continent_code_unique_constraint IF NOT EXISTS
# FOR (c:Continent) REQUIRE c.code IS UNIQUE;

# CREATE CONSTRAINT country_code_unique_constraint IF NOT EXISTS
# FOR (c:Country) REQUIRE c.code IS UNIQUE;

# CREATE INDEX city_name_idx IF NOT EXISTS
# FOR (c:City) ON (c.name);

# CREATE INDEX country_name_idx IF NOT EXISTS
# FOR (c:Country) ON (c.name);

# SHOW INDEXES

In [ ]:
# Examplary queries

# MATCH(n)
# RETURN(n)

# MATCH (c:Continent {code:'EU'})-[:HAS_COUNTRY]->(:Country)-[:HAS_CITY]->(ci:City)
# RETURN ci.name AS CityName;

# MATCH (c:Continent)-[hco:HAS_COUNTRY]->(p:Country)-[hci:HAS_CITY]->(m:City)
# WHERE c.name = 'North America'
# RETURN c.name as continent, p.name as country, collect(m.name) as city, sum(m.areaKm2) as total_area
# ORDER BY total_area DESC

# MATCH (c:Continent)-[hco:HAS_COUNTRY]-(p:Country)-[hci:HAS_CITY]-(m:City)
# WHERE c.name = 'North America'
# WITH c.name as continent, p.name as country, collect(m.name) as city, sum(m.areaKm2) as total_area, count(m) as city_count
# WHERE city_count > 1
# RETURN continent, country, collect(city) as city, total_area
# ORDER BY total_area DESC;

# MATCH (continent:Continent)-[hco:HAS_COUNTRY]->(country:Country)-[hci:HAS_CITY]->(city:City)
# WHERE continent.code IN ['NA', 'EU']
# WITH continent.name as Continent, country.name as Country, collect(city.name) as City, sum(city.areaKm2) as SumArea, count(city.name) as CountCity
# WHERE SumArea > 1000 and CountCity >=3
# RETURN Continent, Country, City, SumArea
# ORDER BY Continent ASC, Country DESC;

In [7]:
import os
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "test12345"))

driver.verify_connectivity()

cypher_query = """
MATCH (c:Continent)-[:HAS_COUNTRY]->(p:Country)-[:HAS_CITY]->(m:City)
WHERE c.name = $continent_name
WITH c.name as continent, p.name as country, collect(m.name) as city, sum(m.areaKm2) as total_area, count(m) as city_count
WHERE city_count > $min_city_count
RETURN continent, country, city, total_area
ORDER BY total_area DESC
"""

records, summary, keys = driver.execute_query(
    cypher_query, continent_name="North America", min_city_count=1
)

for record in records:
   print(f"Cities: {record['city']}, Total area: {record['total_area']}")

driver.close()

Cities: ['Los Angeles', 'Chicago', 'New York City'], Total area: 2676
Cities: ['Vancouver', 'Toronto'], Total area: 745
